In [1]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pprint import pprint
from services.db import MongoDB_one_zero_four


plt.rcParams['font.family'] = ['Microsoft JhengHei']
plt.rcParams['axes.unicode_minus'] = False

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 10)
pd.set_option("display.width", 100)

In [2]:
db = MongoDB_one_zero_four()

condition = {"header.jobName": {"$regex": "python", "$options": "i"}}
projection = {"header": 1, "condition": 1, "_id": 1}
result = db.select_from_bronze(condition, projection)

In [19]:
test = []
for row in result:
    data = row["condition"]["specialty"]
    data.append({"job_id":row["_id"]})
    test.append(data)

In [20]:
print(test[0])

[{'code': '12001002016', 'description': 'Github'}, {'code': '12001003045', 'description': 'Python'}, {'code': '12001003086', 'description': 'PyTorch'}, {'code': '12001002018', 'description': 'Git'}, {'code': '12001001007', 'description': 'Linux'}, {'job_id': '8uzq4'}]


In [21]:
"""
轉換成 df 時是以每個 list 作為一個 column, 所以 df 是以 0, 1, ... list index 作為 col name, 
stack 之後因為 series 只有一行, 所以變成row[row] 的格式? 這種格式經過 df.to_list() 會以[row[row], row[row]] 這樣被拉平? 
會像是 [{}, {}, {}]? 有點神奇, 經過這些操作後撫平了一層巢狀結構
"""
"""
處理這種結構: [[dict, dict], [dict, dict], [dict, dict]]
"""
df = pd.DataFrame(test)
series = df.stack()
df_clean = pd.DataFrame(series.to_list())
print(df_clean)

# print("------------")
# series = pd.Series(test)
# print(series)

            code description job_id
0    12001002016      Github    NaN
1    12001003045      Python    NaN
2    12001003086     PyTorch    NaN
3    12001002018         Git    NaN
4    12001001007       Linux    NaN
..           ...         ...    ...
414  12001005023         AWS    NaN
415  12001006013        HTML    NaN
416  12001006017  JavaScript    NaN
417  12001006032         CSS    NaN
418          NaN         NaN  7wutg

[419 rows x 3 columns]


In [3]:
"""
先找出希望用資料回答什麼問題, 才知道要寫什麼邏輯來實現
1. 在 header.jobName 包含 Python or 後端 or engineer or 資料工程的職位中, 
    1. tool (specialty) 的分布
    2. skill 的分布
"""
condition = {
    "header.jobname":{
        "$regex":r"python|後端|engineer|資料工程",
        "$options":"i"
}}

projection = {"_id":1, "header":1, "condition":1}
result2 = db.select_from_bronze(condition, projection)

In [4]:
with open("Data_from_bronze.json", "w", encoding="utf-8") as f:
    json.dump(result, f, indent=4, ensure_ascii=False)